# Day 1 · Unit 1.1 — What an LLM Does
*100 Days of AI Engineering (CampusX) · my notes + experiments, in simple words*

**Mental model:** `Text → Tokens → Numbers → Model layers → Next-token probabilities → pick one → repeat`

**Goal of today:** explain in two sentences how an LLM produces an answer, and why it can sound confident while being wrong.

## Part 1 — The 5 ideas in simple words

### 1. Next-token prediction
An LLM is a very advanced "what comes next?" machine. You give it some text. It gives a **probability for every possible next token**. One token is picked, added to the text, and the same thing repeats.

```
"The capital of France is"  →  " Paris" 82% | " the" 4% | " a" 2% ...
pick " Paris"  →  "The capital of France is Paris"  →  next token...
```
- A **token** is a small piece of text (a word, part of a word, or punctuation).
- The pick is **random according to the probabilities**. That is why the same prompt can give different answers.

### 2. What an LLM is NOT
- **Not a database:** it does not store facts in rows and look them up.
- **Not a search engine:** it does not search anything while answering (unless you give it a search tool).
- **Not a fixed program:** same input can give different outputs.

It only produces text that *looks likely*. Likely is not the same as true.

### 3. The path through the model
1. **Tokenizer** cuts text into tokens and gives each an ID number.
2. **Embeddings** turn each ID into a list of numbers that carries its meaning.
3. **Layers** (many stacked blocks) improve those numbers by looking at the other tokens around.
4. **Output scores** → **softmax** turns them into probabilities (all add up to 1).
5. **Sampling** picks one token.

### 4. What parameter counts mean
"8B" or "70B" = **8 or 70 billion parameters** = the learned numbers inside the model.
- At 16-bit, each parameter takes 2 bytes → **8B ≈ 16 GB**, **70B ≈ 140 GB** of memory.
- Bigger often means more capable, but **not always better for your task**. A small model can be cheaper, faster and good enough. You find out by testing.

### 5. Why chat models behave the way they do
- **Pretraining:** learn to predict the next token on huge text → a *base model* that just continues text.
- **Instruction tuning:** trained on examples of "question → good answer" → it starts answering.
- **Human feedback training (RLHF):** people rank answers → it learns to be polite, follow instructions, refuse some things and hedge.

A chat model is still next-token prediction underneath. The "assistant personality" is extra training on top.

## Experiment A — See the probabilities (GPT-2 on my own machine/Colab)

**What this code does, step by step**
1. Loads **GPT-2**, a small old model (~124M parameters), and its tokenizer.
2. Turns my prompt into tokens and IDs, and prints them.
3. Asks the model for scores for the **next token**, converts them into probabilities with `softmax`, and prints the top 10.
4. Defines `generate()`, a loop that picks one token, adds it, and repeats. **This loop is the whole idea of an LLM.**
5. Generates the same prompt 3 times.

In [35]:
# pip install transformers torch
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

prompt = "The most amazing people are"
ids = tok(prompt, return_tensors="pt")
print(tok.convert_ids_to_tokens(ids.input_ids[0]), ids.input_ids[0].tolist())

with torch.no_grad():
    logits = model(**ids).logits[0, -1]       # scores for the next token
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"{tok.decode(i)!r:14} {p:.3f}")

# Manual generation loop = the whole idea of an LLM
def generate(prompt, steps=15, temperature=1.0):
    ids = tok(prompt, return_tensors="pt").input_ids
    for _ in range(steps):
        with torch.no_grad():
            logits = model(ids).logits[0, -1] / temperature
        nxt = torch.multinomial(torch.softmax(logits, -1), 1)
        ids = torch.cat([ids, nxt.unsqueeze(0)], dim=1)
    return tok.decode(ids[0])

for _ in range(3):
    print(generate("The most amazing people are"))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

['The', 'Ġmost', 'Ġamazing', 'Ġpeople', 'Ġare'] [464, 749, 4998, 661, 389]
' the'         0.112
' those'       0.082
' people'      0.053
' in'          0.040
' still'       0.022
' not'         0.018
' here'        0.012
' always'      0.012
' going'       0.011
' coming'      0.010
The most amazing people are Ghanaians, who have often found it quite challenging to take crowd control measures
The most amazing people are Muslims. This threat of terrorism is greatest in the land where we live,"
The most amazing people are the Republican Party establishment. That is why the Republicans found a place in the


### What my output tells me

- **Tokens:** `['The', 'Ġmost', 'Ġamazing', 'Ġpeople', 'Ġare']` is 5 tokens with IDs `[464, 749, 4998, 661, 389]`. `Ġ` just means "this token starts with a space". The model never sees words, only these numbers.
- **Flat distribution:** the best next token `' the'` has only **11.2%**, then `' those'` 8.2%, `' people'` 5.3%. The top 10 together are only about **37%**, so the other ~63% is spread over thousands of other tokens. For an open prompt like this, the model has no strong favourite.
- **Why 3 different sentences:** each run randomly picked a different token (`Ghanaians`, `Muslims`, `the Republican Party...`). After one token is chosen, everything after it follows that choice, so the sentences go in totally different directions. There is no plan and no opinion, only sampling.
- **Fluent but meaningless:** every output is grammatical and sounds sure of itself, but the content is random web-style text. GPT-2 is tiny and trained on raw internet text, so it copies internet patterns, not facts. Bigger chat models are much better, but the mechanism is the same.
- **Compare:** for `"The capital of France is"` the top token should be much higher. When the training data clearly points to one answer, the distribution is sharp. When it does not, it is flat.

### Optional extra (run this to see sharp vs flat, and what temperature does)

In [ ]:
# Sharp vs flat distribution
for prompt in ["The capital of France is", "The most amazing people are"]:
    ids = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**ids).logits[0, -1], -1)
    top = torch.topk(probs, 10)
    print(f"{prompt!r} | top-1: {tok.decode(top.indices[0])!r} {top.values[0]:.2f} | top-10 total: {top.values.sum():.2f}")

# Temperature: low = safe/repetitive, high = wild
for t in [0.2, 1.0, 1.5]:
    print(f"\ntemp={t}")
    for _ in range(3):
        print(generate("The most amazing people are", temperature=t))

## Experiment B — The official "Try it": same prompt × 5 (Groq, `openai/gpt-oss-20b`)

**What this code does**
1. `ask()` sends one prompt to the model and returns the answer text. `temperature=1.0` means normal randomness. `reasoning_effort="low"` keeps the model's hidden thinking short, and `max_completion_tokens=600` leaves room for it.
2. Three prompts test three situations: a **factual** question, a **creative** task, and a **fake** thing that does not exist.
3. Each prompt is sent **5 times** so I can see what changes between runs.

*(Needs the `client` cell with my Groq key from Colab Secrets. Never paste the key into the notebook.)*

In [34]:
MODEL = "openai/gpt-oss-20b"
print("Using model:", MODEL)   # confirm this prints gpt-oss-20b

def ask(prompt, temperature=1.0, model=MODEL):
    r = client.chat.completions.create(
        model=model,
        temperature=temperature,
        reasoning_effort="low",
        max_completion_tokens=600,
        messages=[{"role": "user", "content": prompt}],
    )
    return (r.choices[0].message.content or "[empty - raise max_completion_tokens]").strip()

prompts = {
    "factual":  "Who wrote the novel 'Udaas Naslain' and when was it published?",
    "creative": "Write a one-line tagline for a Karachi chai startup.",
    "fake":     "Summarize the 2019 paper 'Quantum Banana Networks' by Dr. Hina Farooq.",
}

for name, p in prompts.items():
    print(f"\n=== {name.upper()}: {p}")
    for i in range(5):
        print(f"[{i+1}] {ask(p)}\n")

Using model: openai/gpt-oss-20b

=== FACTUAL: Who wrote the novel 'Udaas Naslain' and when was it published?
[1] **Author:** Kavita Shankar  
**Publication year:** 2003

[2] **Udaas Naslain** is a celebrated Urdu novel written by **Shahid Ghafoor**. It was first published in **1990**.

[3] **Answer**

The novel *Udaas Naslain* (Urdu: उदास नस्लें) was written by the Pakistani novelist **Sajjad Rizvi**. It was first published in **1996**.

[4] **Answer**

*Udaas Naslain* (meaning “The Sad Generation”) is a novel written by **Shahid Hasan**. It was first published in **1974**.

[5] **Answer**

- **Author:** *Udaas Naslain* was written by **Syeda Shaukat Rizvi**.  
- **Publication Year:** The novel was first published in **1989**.

(If you’re looking for additional details such as the publishing house, place of publication, or any later editions, just let me know!)


=== CREATIVE: Write a one-line tagline for a Karachi chai startup.
[1] "Brewing Karachi’s heartbeat—one steaming cup at a ti

### What my results tell me

**FACTUAL — "Who wrote *Udaas Naslain*?"**
The 5 runs gave **5 different authors and 5 different years** (Kavita Shankar 2003, Shahid Ghafoor 1990, Sajjad Rizvi 1996, Shahid Hasan 1974, Syeda Shaukat Rizvi 1989). As far as I know, the real author is **Abdullah Hussein** (published 1963), so the model was wrong all 5 times. (Worth double-checking this yourself.) Two lessons:
- The model does not have a clean stored fact. It *makes up a likely-looking name*, and sampling picks a different one each time.
- **Every answer sounded equally sure.** Different answers across runs is a warning sign: if the same question gives different facts, don't trust any of them.

**CREATIVE — Karachi chai tagline**
5 different taglines, and that is fine. There is no single correct answer, so variety is a feature here. This is why high temperature suits brainstorming (Unit 1.4).

**FAKE — "Quantum Banana Networks" by Dr. Hina Farooq**
This paper does not exist (I made it up), yet the model wrote a full, professional summary with headings, a table, "core ideas" and "phase transitions". It did **not** say "I can't find this paper". The model's job is to continue the text in a believable way, so it filled the gap with believable nonsense. This is **hallucination**, and it is the exact "confident but wrong" behaviour of today's outcome.

**Big takeaway:** fluent and confident does **not** mean correct. For facts, give the model the real documents (RAG), use tools, or check the output.

## Self-check answers

**1. Why does the same prompt give different answers?**
At each step the model gives *probabilities*, and one token is picked randomly according to them. Different picks lead to different texts, and small early differences grow. (My Experiment A showed 3 different sentences from the same prompt.)

**2. Is an LLM a database? Why or why not?**
No. A database stores exact records and returns the same result every time. An LLM stores patterns, squeezed into billions of numbers, and generates text that is *likely*. My Experiment B proves it: asking the same factual question 5 times gave 5 different authors.

**3. What does "70B" count, and roughly how much memory does it need at 16-bit?**
It counts the **parameters** (the learned numbers) = 70 billion. At 16-bit each takes 2 bytes, so 70B × 2 bytes ≈ **140 GB**. (Quantization to 8-bit or 4-bit makes it much smaller.)

**4. What is the difference between a base model and a chat model?**
A **base model** only continues text, so it may answer a question with more questions. A **chat model** is a base model trained further on instruction examples and human feedback, so it answers, follows instructions, is polite, refuses some requests and hedges.

**5. Why does a hallucination often sound *more* confident than an honest "I don't know"?**
Training rewards the *most likely, fluent continuation*, not truth. A confident, well-written answer is very likely text, and "I don't know" is rare in training data. The model has no built-in fact-checker, so a wrong answer is written in the same smooth style as a right one. Human-feedback training can also push models toward answers that *look* helpful and sure.

---

## ✅ Day 1 Outcome (two sentences)

> An LLM turns text into tokens and numbers, runs them through layers of learned parameters to get a probability for every possible next token, picks one, adds it, and repeats. Because it is trained to produce *likely-sounding* text and not to check facts, it can write fluent, confident answers that are completely wrong.

**Next:** Unit 1.2 — Tokens, Context and Latency (why Hindi, code and numbers cost more tokens than plain English).